# Baseline Colab — MOT15 + MOT16

Генерация `.npy` (det + ReID) и прогон оригинального DeepSORT на **6 sequences**.  
Оба набора генерируются одинаково через `generate_detections.py` → `MOT15_train/` и `MOT16_train/`.

In [ ]:
DRIVE_RESOURCES = "/content/drive/MyDrive/DeepSORT/resources"

from google.colab import drive
drive.mount("/content/drive")

import os
assert os.path.isdir(DRIVE_RESOURCES), f"Папка не найдена: {DRIVE_RESOURCES}"

# Клонируем код (HTTPS, без SSH)
if not os.path.isdir("/content/DeepSORT_Project_CV"):
    !git clone https://github.com/Valeriia-Reznik-Dev/DeepSORT_Project_CV.git /content/DeepSORT_Project_CV

%cd /content/DeepSORT_Project_CV
!git pull

# Подключаем ваши данные с Drive
if os.path.islink("resources"):
    os.remove("resources")
elif os.path.isdir("resources"):
    !rm -rf resources
!ln -s "{DRIVE_RESOURCES}" resources

print("resources ->", os.path.realpath("resources"))

In [ ]:
!pip install -q "numpy>=2.0" opencv-python scipy pyyaml

import numpy as np
print("NumPy", np.__version__)
assert np.__version__.split(".")[0] >= "2", (
    "numpy<2 обнаружен. Перезапустите рантайм (Runtime -> Restart session) и выполните ячейку заново."
)

import tensorflow as tf
print("TensorFlow", tf.__version__)
print("GPU", tf.config.list_physical_devices("GPU"))


In [ ]:
import os

MOT15_SEQUENCES = ["TUD-Campus", "TUD-Stadtmitte", "KITTI-17", "PETS09-S2L1"]
MOT16_SEQUENCES = ["MOT16-09", "MOT16-11"]
DRIVE_RESOURCES = "/content/drive/MyDrive/DeepSORT/resources"

paths = {
    "model": os.path.join(DRIVE_RESOURCES, "networks", "mars-small128.pb"),
    "mot15_dir": "resources/detections/MOT15/train",
    "mot16_dir": "resources/detections/MOT16/train",
    "mot15_npy_dir": "resources/detections/MOT15_train",
    "mot16_npy_dir": "resources/detections/MOT16_train",
}

assert os.path.isfile(paths["model"]), f"Missing: {paths['model']}"

for seq in MOT15_SEQUENCES:
    assert os.path.isdir(os.path.join(paths["mot15_dir"], seq)), f"Missing MOT15 video: {seq}"
for seq in MOT16_SEQUENCES:
    assert os.path.isdir(os.path.join(paths["mot16_dir"], seq)), f"Missing MOT16 video: {seq}"

print("OK: model + MOT15/MOT16 videos on Drive")

In [ ]:
import os
import time
from google.colab import drive

DRIVE_ROOT = "/content/drive"
DRIVE_RESOURCES = "/content/drive/MyDrive/DeepSORT/resources"
SRC_MODEL = os.path.join(DRIVE_RESOURCES, "networks", "mars-small128.pb")
MODEL = "/content/mars-small128.pb"


def ensure_drive() -> None:
    """Remount Drive if the FUSE mount is dead (OSError 107)."""
    try:
        with open(SRC_MODEL, "rb") as f:
            f.read(1)
    except OSError as exc:
        print(f"Drive mount unhealthy ({exc}); remounting ...")
        drive.mount(DRIVE_ROOT, force_remount=True)


def copy_from_drive(src: str, dst: str, *, chunk_size: int = 1024 * 1024, retries: int = 5) -> None:
    """Copy a Drive file to local disk; remount Drive on FUSE disconnect (errno 107)."""
    ensure_drive()
    src_size = os.path.getsize(src)
    if os.path.isfile(dst) and os.path.getsize(dst) == src_size:
        print(f"Already cached: {dst}")
        return

    last_err = None
    for attempt in range(1, retries + 1):
        try:
            with open(src, "rb") as fsrc, open(dst, "wb") as fdst:
                while True:
                    chunk = fsrc.read(chunk_size)
                    if not chunk:
                        break
                    fdst.write(chunk)
            if os.path.getsize(dst) == src_size:
                print(f"Copied {src_size / 1e6:.1f} MB -> {dst}")
                return
            last_err = OSError(f"size mismatch: got {os.path.getsize(dst)}, expected {src_size}")
        except OSError as exc:
            last_err = exc
            if os.path.isfile(dst):
                os.remove(dst)
            print(f"Drive copy failed ({exc}); remount + retry {attempt}/{retries} in 5s ...")
            time.sleep(5)
            ensure_drive()
            src_size = os.path.getsize(src)

    raise OSError(
        f"Could not copy {src} -> {dst}: {last_err}. "
        "Remount Drive: Runtime -> Restart session, rerun from cell 1, "
        "or drive.mount('/content/drive', force_remount=True)."
    )


copy_from_drive(SRC_MODEL, MODEL)
print("MODEL ->", MODEL)

# Кадры ниже читаются через симлинк resources -> Drive; убедимся, что монтирование живо.
ensure_drive()

# MOT15 + MOT16: один pipeline, generate_detections.py
!python tools/generate_detections.py \
    --model="{MODEL}" \
    --mot_dir=resources/detections/MOT15/train \
    --output_dir=resources/detections/MOT15_train

!python tools/generate_detections.py \
    --model="{MODEL}" \
    --mot_dir=resources/detections/MOT16/train \
    --output_dir=resources/detections/MOT16_train

In [ ]:
import numpy as np

ALL_SEQUENCES = [
    (paths["mot15_npy_dir"], MOT15_SEQUENCES),
    (paths["mot16_npy_dir"], MOT16_SEQUENCES),
]

for npy_dir, seqs in ALL_SEQUENCES:
    print(f"--- {npy_dir}")
    for seq in seqs:
        path = os.path.join(npy_dir, f"{seq}.npy")
        assert os.path.isfile(path), f"Missing: {path}"
        print(seq, np.load(path).shape)

print("OK: all 6 .npy ready")

In [ ]:
!python scripts/run_baseline.py


In [ ]:
# Этап 2: TrackEval HOTA на baseline results
!pip install -q "trackeval==1.3.0"

!python scripts/run_eval.py

## Detector F1 (YOLO + NanoDet + MMDet)

Сравнение детекций с GT: Precision / Recall / F1 для **всех трёх** детекторов.  
Сначала smoke test (`--max-frames 30`), потом полный прогон без флага.

In [ ]:
!python scripts/setup_detectors_colab.py
# веса NanoDet + RTMDet (если ещё нет в resources/models/)
!python scripts/download_detector_models.py

# Полный прогон: Precision / Recall / F1 + FPS по всем 6 видео.
!python scripts/run_detector_eval.py --detector yolo nanodet mmdet

## ReID clustering metrics (OSNet + ResNet50-IBN + fast-reid)

Standalone-оценка на **GT pedestrian crops**: Silhouette, Calinski–Harabasz, Fowlkes–Mallows.
Сначала smoke test (`--max-frames 30 --max-samples 500`), потом полный прогон.

In [ ]:
!git pull
!python scripts/setup_reid_colab.py
!python scripts/download_reid_models.py

# smoke test — все 3 ReID модели
!python scripts/run_reid_eval.py --model osnet resnet50_ibn fastreid --max-frames 30 --max-samples 500

# полный прогон
!python scripts/run_reid_eval.py --model osnet resnet50_ibn fastreid

## Модифицированный DeepSORT (детектор + ReID) → HOTA

Live-пайплайн: детектор → ReID-дескрипторы → ядро DeepSORT (Kalman + matching cascade) → MOT-txt.


In [ ]:
NAME = "yolo_osnet"
# Прогон трекера по 6 видео (печатает FPS по каждому).
!python scripts/run_tracker.py --detector yolo --reid osnet --tracker-name {NAME}

# HOTA модифицированного трекера 
!python scripts/run_eval.py --tracker-name {NAME} --results-dir results/tracking/{NAME}

## Перебор связок (detector × ReID) → сводная таблица

Несколько комбинаций, где считается HOTA по каждому видео + средний FPS, сравнивает с baseline и отмечает лучшую real-time связку (≥5 FPS)

In [ ]:
!python scripts/run_sweep.py --combos yolo:osnet yolo:resnet50_ibn nanodet:osnet mmdet:osnet yolo_seg:osnet:seg

import pandas as pd
pd.read_csv("results/tracking/sweep_summary.csv")

## Оверлеи лучшего решения

Рендерит видео-оверлеи для лучшей связки в `overlays/best/` (для сдачи рядом с `overlays/original/`).
Поменяйте `NAME`, если лучшая связка другая.

In [ ]:
NAME = "yolo_osnet"
# Оверлеи лучшего решения -> overlays/best/<seq>.avi и .mp4
!python scripts/run_overlays.py \
    --results-dir results/tracking/{NAME} \
    --overlays-dir overlays/best \
    --convert_h264

import os
print("overlays/best:", sorted(os.listdir("overlays/best")))

## Этап 6 — сегментация → маски убирают фон перед ReID

Доступны три сегментатора (выбор до запуска через `--detector`):
- `yolo_seg` — YOLOv8-seg (ultralytics, real-time)
- `detectron2_seg` — Mask R-CNN (detectron2, quality mode)
- `smp_seg` — DeepLabV3+ (segmentation_models.pytorch)

С флагом `--mask-background` фон зануляется до извлечения ReID-дескрипторов.
Для detectron2/smp сначала: `python scripts/setup_segmentation_colab.py`

In [ ]:
# Опционально: detectron2 + smp (Mask R-CNN / DeepLabV3+)
!python scripts/setup_segmentation_colab.py
!python scripts/setup_segmentation_colab.py --download-smp-weights

NAME = "yolo_seg_osnet_seg"  # real-time: YOLOv8-seg + osnet + mask background
!python scripts/run_tracker.py --detector yolo_seg --reid osnet --mask-background --tracker-name {NAME}
!python scripts/run_eval.py --tracker-name {NAME} --results-dir results/tracking/{NAME}

# quality mode (медленнее): Mask R-CNN
# NAME = "detectron2_seg_osnet_seg"
# !python scripts/run_tracker.py --detector detectron2_seg --reid osnet --mask-background --tracker-name {NAME}

## Этап 7 — Additional: автономная база идентичностей (body ReID)

Онлайн-галерея (`IdentityDatabase`) + kNN lookup + majority-vote по окну + conflict resolution.
Встроена в live-трекер: `--identity` (sidecar `*_identity.csv`: track_id → identity).
Оценка на GT-кропах: Fowlkes–Mallows / ARI / V-measure для `db_raw` и `resolved`.
Параметры: `configs/identity.yaml` или CLI `--radius --k --representation --window`.

In [ ]:
# Standalone eval на GT-кропах (подбор параметров до интеграции)
!python scripts/run_identity_eval.py --reid osnet \
    --radius 0.3 --k 1 --representation centroid --window 30 --conflict-policy distance

# Live-трекер с identity DB (лучшая real-time связка + --identity)
NAME = "yolo_osnet_id"
!python scripts/run_tracker.py --detector yolo --reid osnet --identity --tracker-name {NAME}
# Sidecar: results/tracking/{NAME}/<seq>_identity.csv

# kNN-режим галереи
!python scripts/run_identity_eval.py --reid osnet --representation knn --k 5 --radius 0.25

# Эволюция параметров, per-video, GT-bbox ReID

- sweep по параметрам трекера (`max_cosine_distance`, `min_confidence`) → HOTA + FPS;
- sweep по параметрам identity DB (`radius`, `representation`) → FM/ARI;
- финальный прогон лучшей связки с **per-video** параметрами (`configs/tracker_params.yaml`);
- оценка ReID на **GT-боксах** (детектор выключен) → HOTA, изолируем качество ReID/ассоциации.

In [ ]:
!git pull

# Эволюция параметров трекера (HOTA по каждому видео + FPS) на лучшей связке yolo+osnet
!python scripts/run_param_sweep.py --param max_cosine_distance --values 0.1 0.2 0.3 0.4 --detector yolo --reid osnet
!python scripts/run_param_sweep.py --param min_confidence --values 0.2 0.3 0.4 0.5 --detector yolo --reid osnet

import pandas as pd
display(pd.read_csv("results/param_sweep/sweep_max_cosine_distance.csv"))
display(pd.read_csv("results/param_sweep/sweep_min_confidence.csv"))

In [ ]:
# Эволюция параметров identity DB (FM/ARI на GT-кропах)
!python scripts/run_identity_sweep.py --param radius --values 0.2 0.25 0.3 0.35 0.4 --reid osnet
!python scripts/run_identity_sweep.py --param representation --values centroid knn --reid osnet

import pandas as pd
display(pd.read_csv("results/identity_sweep/sweep_radius.csv"))
display(pd.read_csv("results/identity_sweep/sweep_representation.csv"))

In [ ]:
# Финальный прогон лучшей связки с per-video параметрами (configs/tracker_params.yaml)
NAME = "yolo_osnet_pv"
!python scripts/run_tracker.py --detector yolo --reid osnet \
    --params-config configs/tracker_params.yaml --tracker-name {NAME}
!python scripts/run_eval.py --tracker-name {NAME} --results-dir results/tracking/{NAME}

In [ ]:

# Сравните ReID-модели по HOTA без влияния ошибок детектора.
for REID in ["osnet", "resnet50_ibn", "fastreid"]:
    !python scripts/run_tracker.py --gt-detections --reid {REID} --tracker-name gtdet_{REID}
    !python scripts/run_eval.py --tracker-name gtdet_{REID} --results-dir results/tracking/gtdet_{REID}

In [ ]:
!zip -r project_outputs.zip \
    results/baseline/original/ \
    results/tracking/ \
    results/detectors/ \
    results/reid/ \
    results/identity/ \
    results/param_sweep/ \
    results/identity_sweep/ \
    overlays/

from google.colab import files
files.download("project_outputs.zip")
